In [1]:
from pathlib import Path
import os
import sys

import mne
import numpy as np
from mne_icalabel import label_components

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)

os.chdir(project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

%matplotlib qt

c:\Repositorios\msc-eeg-tms-pipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from modules.decode_trigger import decode_8bit_trigger, convert_dict_trigger
from modules.events import get_events_tms_per_task

In [3]:
raw_data = mne.io.read_raw_bdf(r"data/raw/V1.bdf", preload=True)

Extracting BDF parameters from data/raw/V1.bdf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 12594999  =      0.000 ...  2519.000 secs...


In [5]:
emg_ch_names = ["EMG L", "EMG R"]
eog_ch_names = ["EOG"]
raw_data.set_channel_types({ch: "emg" for ch in emg_ch_names})
raw_data.set_channel_types({'EOG':'eog'})

montage = mne.channels.make_standard_montage("standard_1020",head_size='auto')
raw_data.set_montage(montage)

<RawBDF | V1.bdf, 67 x 12595000 (2519.0 s), ~6.29 GiB, data loaded>

In [6]:
raw_data.drop_channels(emg_ch_names)

<RawBDF | V1.bdf, 65 x 12595000 (2519.0 s), ~6.10 GiB, data loaded>

In [7]:
filtered_data = raw_data.copy().filter(l_freq=1, h_freq=100, method='iir', iir_params={'order': 4, 'ftype': 'butter'})

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 1e+02 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 1.00, 100.00 Hz: -6.02, -6.02 dB



In [8]:
freqs_notch = np.arange(60, 241, 60)
filtered_data.notch_filter(freqs=freqs_notch, picks='eeg')

Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower transition bandwidth: 0.50 Hz
- Upper transition bandwidth: 0.50 Hz
- Filter length: 33001 samples (6.600 s)



<RawBDF | V1.bdf, 65 x 12595000 (2519.0 s), ~6.10 GiB, data loaded>

In [ ]:
filtered_data = filtered_data.set_eeg_reference("average")

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.


In [10]:
ica = mne.preprocessing.ICA(random_state=97, max_iter=800, method="infomax")
ica.fit(filtered_data)

Fitting ICA to data using 64 channels (please be patient, this may take a while)
Selecting by non-zero PCA components: 31 components
Computing Infomax ICA


KeyboardInterrupt: 

In [ ]:
ica.plot_sources(filtered_data, show_scrollbars=True)
ica.plot_components(filtered_data)

In [ ]:
ica.plot_overlay(filtered_data, exclude=[0], picks="eeg")

In [ ]:
ic_labels = label_components(filtered_data, ica, method="iclabel")

In [ ]:
print(ic_labels["labels"])
ica.plot_properties(filtered_data, picks=[0, 12], verbose=False)

In [ ]:
labels = ic_labels["labels"]
exclude_idx = [
    idx for idx, label in enumerate(labels) if label not in ["brain", "other"]
]
print(f"Excluding these ICA components: {exclude_idx}")

In [ ]:
reconst_data = filtered_data.copy()
ica.apply(reconst_data, exclude=exclude_idx)

In [ ]:
## Cálculo do Rank Efetivo

In [ ]:
reconst_data = reconst_data.set_eeg_reference("average")

In [ ]:
reconst_data.resample(500, npad="auto")

In [ ]:
events, event_id = mne.events_from_annotations(reconst_data)
event_id = reconst_data.event_id = convert_dict_trigger(event_id, decode_8bit_trigger)

In [ ]:
events_left, events_right, events_bilateral = get_events_tms_per_task(events, event_id)

In [ ]:
epochs_left = mne.Epochs(
    reconst_data,
    events_left,
    tmin=-2,
    tmax=0.05,
    preload=True,
)

epochs_right = mne.Epochs(
    reconst_data,
    events_right,
    tmin=-2,
    tmax=0.05,
    preload=True,
)

epochs_bilateral = mne.Epochs(
    reconst_data,
    events_bilateral,
    tmin=-2,
    tmax=0.05,
    preload=True,
)